# Flood-victim (person) detection from UAV imagery: YOLO12l on SeaDronesSee
Single-class (`person`) detector. Trains on 2x T4 (DDP), supports resume, and writes all
metrics (CSV), figures (PNG + EPS) and weights to `/kaggle/working`.

**Before running:** (1) Add the SeaDronesSee dataset as input, (2) set `DATASET_ROOT` in the config cell,
(3) Settings -> Accelerator: GPU T4 x2, Internet: On (pip + pretrained weights).

In [1]:
# Cell 0: install
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics", "pycocotools"], check=False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 kB 4.9 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-U', 'ultralytics', 'pycocotools'], returncode=0)

In [2]:
   import glob
   print(glob.glob("/kaggle/input/**/last.pt", recursive=True))

[]


In [3]:
# Cell 1: configuration
import os, sys, json, re, shutil, random, time, platform, glob, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# ======================= USER CONFIG (edit here) =======================
DATASET_ROOT   = "/kaggle/input/datasets/ubiratanfilho/sds-dataset/compressed"   # <-- folder that contains annotations/ and images/
ANN_TRAIN      = "annotations/instances_train.json"
ANN_VAL        = "annotations/instances_val.json"
ANN_TEST       = None                                       # official test labels are normally withheld
IMG_DIRS       = ["images/train", "images/val", "images/test"]

MODEL_CANDIDATES = ["yolo12l.pt", "yolo11l.pt"]             # first one that loads is used
IMGSZ          = 960          # 1280 is better for tiny objects but much slower on 2x T4
BATCH          = 8            # total across GPUs (4 per GPU). Lower to 4 if CUDA OOM
BATCH_EVAL     = 8
EPOCHS         = 50
PATIENCE       = 10
WORKERS        = 2
SEED           = 42

# class merging: any COCO category whose name contains one of these becomes "person" (all others ignored)
PERSON_KEYWORDS = ("swimmer", "floater", "person", "human")

# "grouped_resplit": pool train+val, split into train/val/test by filename-prefix groups (seeded)
# "official"       : official train / val; test only if ANN_TEST is given
SPLIT_MODE     = "grouped_resplit"
SPLIT_RATIOS   = (0.70, 0.15, 0.15)

RUN_NAME       = "yolo12l_seadronessee_person"
RESUME_FROM_INPUT = None      # e.g. "/kaggle/input/<prev-notebook-output>/runs/yolo12l_seadronessee_person"
EXTERNAL_IMAGES   = None      # e.g. "/kaggle/input/real-flood-images" -> predictions saved as PNG+EPS

N_PRED_FIGS    = 20
PRED_CONF      = 0.25
MAX_FIG_SIDE   = 1600         # px; keeps EPS files a manageable size
# ======================================================================

N_GPU = torch.cuda.device_count()
DEVICE = list(range(min(2, N_GPU))) if N_GPU else "cpu"
print("GPUs:", N_GPU, "| device:", DEVICE)

WORK = Path("/kaggle/working")
OUT = WORK / "outputs"
CSV_DIR = OUT / "csv"
PNG_DIR = OUT / "figures" / "png"
EPS_DIR = OUT / "figures" / "eps"
DS = WORK / "dataset"
RUNS = WORK / "runs"
RUN_DIR = RUNS / RUN_NAME
for p in (CSV_DIR, PNG_DIR, EPS_DIR, RUNS):
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

GPUs: 2 | device: [0, 1]


In [4]:
# Cell 2: COCO -> YOLO conversion, split, dataset statistics
from PIL import Image

ROOT = Path(DATASET_ROOT)
if not (ROOT / ANN_TRAIN).exists():
    found = glob.glob("/kaggle/input/**/instances_train.json", recursive=True)
    raise FileNotFoundError(
        f"{ROOT / ANN_TRAIN} not found. instances_train.json found at: {found}. "
        "Set DATASET_ROOT to the folder that contains annotations/ and images/."
    )


def build_image_index():
    idx, dup = {}, 0
    for d in IMG_DIRS:
        base = ROOT / d
        if not base.exists():
            continue
        for p in base.rglob("*"):
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                if p.name in idx:
                    dup += 1
                else:
                    idx[p.name] = p
    return idx, dup


IMG_INDEX, n_dup = build_image_index()
print(f"Indexed {len(IMG_INDEX)} images ({n_dup} duplicate basenames ignored)")


def coco_records(json_path, source):
    d = json.load(open(json_path))
    cats = {c["id"]: c["name"] for c in d["categories"]}
    person_ids = {i for i, n in cats.items() if any(k in n.lower() for k in PERSON_KEYWORDS)}
    print(f"[{source}] categories: {cats}")
    print(f"[{source}] merged into 'person': {sorted(cats[i] for i in person_ids)}")
    if not person_ids:
        raise ValueError("No category matched PERSON_KEYWORDS; edit the keywords.")
    anns = defaultdict(list)
    for a in d["annotations"]:
        if a["category_id"] in person_ids and not a.get("iscrowd", 0):
            anns[a["image_id"]].append(a["bbox"])
    recs, missing = [], 0
    for im in d["images"]:
        p = IMG_INDEX.get(Path(im["file_name"]).name)
        if p is None:
            missing += 1
            continue
        W, H = im.get("width"), im.get("height")
        if not W or not H:
            W, H = Image.open(p).size
        boxes = []
        for x, y, w, h in anns.get(im["id"], []):
            x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
            if x2 - x1 >= 1 and y2 - y1 >= 1:
                boxes.append([x1, y1, x2 - x1, y2 - y1])
        recs.append(dict(src=p, name=p.name, W=W, H=H, boxes=boxes, origin=source))
    print(f"[{source}] images: {len(recs)} (missing files: {missing}), person boxes: {sum(len(r['boxes']) for r in recs)}")
    return recs


def group_key(name):
    stem = Path(name).stem
    return re.sub(r"[_\-]?\d+$", "", stem) or stem


def grouped_split(recs, ratios, seed):
    groups = defaultdict(list)
    for r in recs:
        groups[group_key(r["name"])].append(r)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    total = len(recs)
    tgt = dict(zip(("train", "val", "test"), [total * x for x in ratios]))
    cnt = {k: 0 for k in tgt}
    out = {k: [] for k in tgt}
    for k in keys:
        s = max(tgt, key=lambda s: (tgt[s] - cnt[s]) / tgt[s])
        out[s] += groups[k]
        cnt[s] += len(groups[k])
    return out, len(keys)


train_recs = coco_records(ROOT / ANN_TRAIN, "train")
val_recs = coco_records(ROOT / ANN_VAL, "val")

if SPLIT_MODE == "grouped_resplit":
    pooled = {r["name"]: r for r in train_recs + val_recs}
    pooled = [pooled[k] for k in sorted(pooled)]
    splits, n_groups = grouped_split(pooled, SPLIT_RATIOS, SEED)
    print(f"Grouped re-split: {len(pooled)} images in {n_groups} groups")
    if n_groups > 0.5 * len(pooled):
        print("WARNING: filenames do not seem to encode video/sequence ids, so near-duplicate frames may leak "
              "across splits. Consider SPLIT_MODE='official' or grouping with the metadata CSVs.")
elif SPLIT_MODE == "official":
    splits = {"train": train_recs, "val": val_recs, "test": []}
    if ANN_TEST and (ROOT / ANN_TEST).exists():
        splits["test"] = coco_records(ROOT / ANN_TEST, "test")
    else:
        print("No labelled test set in 'official' mode: test evaluation will be skipped.")
else:
    raise ValueError("SPLIT_MODE must be 'grouped_resplit' or 'official'")

if DS.exists():
    shutil.rmtree(DS)


def write_split(name, recs):
    (DS / "images" / name).mkdir(parents=True, exist_ok=True)
    (DS / "labels" / name).mkdir(parents=True, exist_ok=True)
    coco = {"images": [], "annotations": [], "categories": [{"id": 1, "name": "person"}]}
    aid = 1
    for r in sorted(recs, key=lambda r: r["name"]):
        stem = Path(r["name"]).stem
        dst = DS / "images" / name / r["name"]
        if not (dst.exists() or dst.is_symlink()):
            os.symlink(r["src"], dst)
        W, H = r["W"], r["H"]
        lines = [f"0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}" for x, y, w, h in r["boxes"]]
        (DS / "labels" / name / f"{stem}.txt").write_text("\n".join(lines))   # empty file = background image
        iid = int(stem) if stem.isdigit() else stem
        coco["images"].append(dict(id=iid, file_name=r["name"], width=W, height=H))
        for x, y, w, h in r["boxes"]:
            coco["annotations"].append(dict(id=aid, image_id=iid, category_id=1, bbox=[x, y, w, h],
                                            area=w * h, iscrowd=0))
            aid += 1
    json.dump(coco, open(DS / f"gt_{name}.json", "w"))


for k, v in splits.items():
    if v:
        write_split(k, v)

yaml_lines = [f"path: {DS}", "train: images/train", "val: images/val"]
if splits["test"]:
    yaml_lines.append("test: images/test")
yaml_lines += ["names:", "  0: person"]
DATA_YAML = DS / "data.yaml"
DATA_YAML.write_text("\n".join(yaml_lines) + "\n")

stats, manifest = [], []
for k, v in splits.items():
    if not v:
        continue
    areas = np.array([w * h for r in v for _, _, w, h in r["boxes"]]) if any(r["boxes"] for r in v) else np.array([])
    stats.append(dict(
        split=k, images=len(v), images_with_person=sum(1 for r in v if r["boxes"]),
        background_images=sum(1 for r in v if not r["boxes"]), instances=int(len(areas)),
        small_lt32=int((areas < 32 ** 2).sum()), medium_32_96=int(((areas >= 32 ** 2) & (areas < 96 ** 2)).sum()),
        large_gt96=int((areas >= 96 ** 2).sum()),
        median_box_area_px=float(np.median(areas)) if len(areas) else 0.0))
    manifest += [dict(split=k, file=r["name"], origin=r["origin"], n_person=len(r["boxes"])) for r in v]
pd.DataFrame(stats).to_csv(CSV_DIR / "dataset_stats.csv", index=False)
pd.DataFrame(manifest).to_csv(CSV_DIR / "split_manifest.csv", index=False)
print(pd.DataFrame(stats).to_string(index=False))
print("data.yaml ->", DATA_YAML)

Indexed 14227 images (0 duplicate basenames ignored)
[train] categories: {0: 'ignored', 1: 'swimmer', 2: 'boat', 3: 'jetski', 4: 'life_saving_appliances', 5: 'buoy'}
[train] merged into 'person': ['swimmer']
[train] images: 8930 (missing files: 0), person boxes: 37096
[val] categories: {0: 'ignored', 1: 'swimmer', 2: 'boat', 3: 'jetski', 4: 'life_saving_appliances', 5: 'buoy'}
[val] merged into 'person': ['swimmer']
[val] images: 1547 (missing files: 0), person boxes: 6206
Grouped re-split: 10477 images in 10477 groups
split  images  images_with_person  background_images  instances  small_lt32  medium_32_96  large_gt96  median_box_area_px
train    7333                5736               1597      30084       14358         14449        1277              1088.0
  val    1572                1199                373       6438        2987          3171         280              1120.0
 test    1572                1249                323       6780        3179          3337         264        

In [5]:
# Cell 3: train (with automatic resume)
import ultralytics
from ultralytics import YOLO

try:
    from ultralytics import settings
    settings.update({"wandb": False, "comet": False, "clearml": False, "mlflow": False})
except Exception:
    pass

LAST = RUN_DIR / "weights" / "last.pt"
BEST = RUN_DIR / "weights" / "best.pt"

if RESUME_FROM_INPUT and not LAST.exists():
    print("Copying previous run from", RESUME_FROM_INPUT)
    shutil.copytree(RESUME_FROM_INPUT, RUN_DIR, dirs_exist_ok=True)

train_args = dict(
    data=str(DATA_YAML), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
    workers=WORKERS, seed=SEED, deterministic=False, project=str(RUNS), name=RUN_NAME, exist_ok=True,
    pretrained=True, single_cls=True, amp=True, cache=False, plots=True, val=True, save=True,
    optimizer="auto", close_mosaic=10, flipud=0.5, fliplr=0.5, mosaic=1.0, scale=0.5, verbose=True,
)

env_info = dict(
    ultralytics=ultralytics.__version__, torch=torch.__version__, python=platform.python_version(),
    gpus=[torch.cuda.get_device_name(i) for i in range(N_GPU)], seed=SEED, split_mode=SPLIT_MODE,
    split_ratios=SPLIT_RATIOS, person_keywords=PERSON_KEYWORDS, train_args={k: str(v) for k, v in train_args.items()},
)
json.dump(env_info, open(OUT / "run_config.json", "w"), indent=2)

t0 = time.time()
try:
    if LAST.exists():
        print("Resuming from", LAST)
        YOLO(str(LAST)).train(resume=True)
    else:
        model, used = None, None
        for w in MODEL_CANDIDATES:
            try:
                model = YOLO(w)
                used = w
                break
            except Exception as e:
                print(f"Could not load {w}: {e}")
        assert model is not None, "No model could be loaded; check internet / ultralytics version."
        print("Training from", used)
        env_info["model"] = used
        json.dump(env_info, open(OUT / "run_config.json", "w"), indent=2)
        model.train(**train_args)
except AssertionError as e:
    print("Resume/train message:", e)
print(f"Training cell finished in {(time.time() - t0) / 3600:.2f} h")
assert BEST.exists(), "best.pt not found; training did not produce weights."

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Training from yolo12l.pt
Ultralytics 8.4.194 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=0.0, deterministic=False, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=Fa

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      13.9G      2.652      1.378      1.033         13        960: 100% ━━━━━━━━━━━━ 917/917 1.2it/s 12:20
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 99/99 2.7it/s 36.8s
                   all       1572       6437      0.597      0.591      0.503      0.152

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      13.9G      2.567      1.285     0.9974         21        960: 100% ━━━━━━━━━━━━ 917/917 1.3it/s 12:09
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 99/99 2.6it/s 37.8s
                   all       1572       6437      0.617      0.571      0.556      0.195

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      13.8G      2.523      1.253     0.9918         14        960: 100% ━━━━━━━

In [6]:
# Cell 4: training curves + per-epoch log (CSV, PNG, EPS)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams.update({"ps.fonttype": 42, "pdf.fonttype": 42, "font.family": "DejaVu Sans",
                     "font.size": 9, "savefig.dpi": 300})

log = pd.read_csv(RUN_DIR / "results.csv")
log.columns = [c.strip() for c in log.columns]
log.to_csv(CSV_DIR / "training_log_per_epoch.csv", index=False)
print("Epochs run:", len(log), "| best epoch (mAP50-95):", int(log["metrics/mAP50-95(B)"].idxmax()) + 1)

fig, axes = plt.subplots(2, 3, figsize=(12, 6.5))
panels = [("train/box_loss", "val/box_loss", "Box loss"), ("train/cls_loss", "val/cls_loss", "Cls loss"),
          ("train/dfl_loss", "val/dfl_loss", "DFL loss"), ("metrics/precision(B)", "metrics/recall(B)", "Precision / Recall"),
          ("metrics/mAP50(B)", None, "mAP@0.5"), ("metrics/mAP50-95(B)", None, "mAP@0.5:0.95")]
for ax, (a, b, t) in zip(axes.ravel(), panels):
    ax.plot(log["epoch"], log[a], label=a.split("/")[-1])
    if b:
        ax.plot(log["epoch"], log[b], label=b.split("/")[-1])
        ax.legend(frameon=False)
    ax.set_title(t)
    ax.set_xlabel("Epoch")
fig.tight_layout()
fig.savefig(PNG_DIR / "training_curves.png", dpi=300)
fig.savefig(EPS_DIR / "training_curves.eps", format="eps", dpi=300)
plt.close(fig)

Epochs run: 50 | best epoch (mAP50-95): 50


In [7]:
# Cell 5: evaluate best.pt on validation and test (CSV outputs)
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

torch.cuda.empty_cache()
best_model = YOLO(str(BEST))
EVAL_DEV = 0 if N_GPU else "cpu"


def coco_metrics(split, pred_json):
    names = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
             "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large"]
    gt = COCO(str(DS / f"gt_{split}.json"))
    dt = gt.loadRes(str(pred_json))
    ev = COCOeval(gt, dt, "bbox")
    ev.evaluate(); ev.accumulate(); ev.summarize()
    return dict(zip(names, [float(x) for x in ev.stats]))


def evaluate(split):
    m = best_model.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH_EVAL, conf=0.001, iou=0.6,
                       device=EVAL_DEV, plots=True, save_json=True, project=str(OUT / "eval"), name=split,
                       exist_ok=True, verbose=True)
    b = m.box
    n_img = next(s["images"] for s in stats if s["split"] == split)
    n_inst = next(s["instances"] for s in stats if s["split"] == split)
    p, r = float(b.mp), float(b.mr)
    row = dict(split=split, images=n_img, instances=n_inst, precision=p, recall=r,
               f1=2 * p * r / (p + r + 1e-16), mAP50=float(b.map50), mAP75=float(b.map75), mAP50_95=float(b.map),
               preprocess_ms=m.speed["preprocess"], inference_ms=m.speed["inference"], postprocess_ms=m.speed["postprocess"])
    try:
        f1c = np.asarray(b.f1_curve).mean(0)
        row["best_f1_confidence"] = float(np.asarray(b.px)[f1c.argmax()])
    except Exception:
        pass
    pd.DataFrame([row]).to_csv(CSV_DIR / f"metrics_{split}_summary.csv", index=False)

    pc = []
    for i, ci in enumerate(b.ap_class_index):
        pi, ri, ap50, ap = b.class_result(i)
        pc.append(dict(class_id=int(ci), class_name=m.names[int(ci)], precision=float(pi), recall=float(ri),
                       f1=float(2 * pi * ri / (pi + ri + 1e-16)), mAP50=float(ap50), mAP50_95=float(ap)))
    pd.DataFrame(pc).to_csv(CSV_DIR / f"metrics_{split}_per_class.csv", index=False)

    try:
        cm = coco_metrics(split, OUT / "eval" / split / "predictions.json")
        pd.DataFrame([dict(split=split, **cm)]).to_csv(CSV_DIR / f"metrics_{split}_coco_size.csv", index=False)
    except Exception as e:
        print(f"COCO size-based metrics skipped for {split}: {e}")
    return row


results = [evaluate("val")]
if splits["test"]:
    results.append(evaluate("test"))
else:
    print("No labelled test split; skipping test evaluation.")
pd.DataFrame(results).to_csv(CSV_DIR / "metrics_val_test_combined.csv", index=False)
print(pd.DataFrame(results).round(4).to_string(index=False))

Ultralytics 8.4.194 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 95.1 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.0 ms, read: 981.1±233.1 MB/s, size: 692.8 KB)
val: Scanning /kaggle/working/dataset/labels/val.cache... 1572 images, 373 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1572/1572 366.3Mit/s 0.0s
val: /kaggle/working/dataset/images/val/9566.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 197/197 1.5it/s 2:10
                   all       1572       6437      0.844      0.829      0.819      0.368
Speed: 1.2ms preprocess, 77.0ms inference, 0.0ms loss, 0.7ms postprocess per image
Saving /kaggle/working/outputs/eval/val/predictions.json...

Evaluating faster-coco-eval mAP...
requirements: Ultralytics requirement ['faster-coco-eval>=1.6.7'] not found, attempting AutoUpdate...
Using Python 3.13.15 enviro

In [8]:
# Cell 6: prediction figures (PNG + EPS, vector boxes/text over embedded raster)
def _panel(ax, img, boxes, color, title, scores=None):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=1.4))
        if scores is not None:
            ax.text(x1, max(y1 - 3, 0), f"{scores[i]:.2f}", color="white", fontsize=6, va="bottom",
                    bbox=dict(facecolor=color, edgecolor="none", pad=0.8))


def save_prediction_figures(model, items, tag, conf=PRED_CONF):
    for k, (path, gt) in enumerate(items):
        res = model.predict(str(path), imgsz=IMGSZ, conf=conf, iou=0.6, device=EVAL_DEV, verbose=False)[0]
        pb = res.boxes.xyxy.cpu().numpy()
        ps = res.boxes.conf.cpu().numpy()
        im = Image.open(path).convert("RGB")
        W, H = im.size
        s = min(1.0, MAX_FIG_SIDE / max(W, H))
        im = im.resize((max(1, int(W * s)), max(1, int(H * s))), Image.LANCZOS)
        pb = pb * s
        ncol = 2 if gt is not None else 1
        fig, axes = plt.subplots(1, ncol, figsize=(7 * ncol, 7 * H / W + 0.4))
        axes = np.atleast_1d(axes)
        if gt is not None:
            g = np.array([[x, y, x + w, y + h] for x, y, w, h in gt]).reshape(-1, 4) * s
            _panel(axes[0], im, g, "#00a651", f"Ground truth ({len(g)})")
        _panel(axes[-1], im, pb, "#e41a1c", f"Prediction ({len(pb)}, conf>={conf})", ps)
        fig.tight_layout()
        stem = f"{tag}_{k:03d}_{Path(path).stem}"
        fig.savefig(PNG_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
        fig.savefig(EPS_DIR / f"{stem}.eps", format="eps", dpi=300, bbox_inches="tight")
        plt.close(fig)


eval_split = "test" if splits["test"] else "val"
pool = [r for r in splits[eval_split] if r["boxes"]]
random.Random(SEED).shuffle(pool)
items = [(r["src"], r["boxes"]) for r in pool[:N_PRED_FIGS]]
save_prediction_figures(best_model, items, eval_split)

if EXTERNAL_IMAGES:
    ext = sorted(p for p in Path(EXTERNAL_IMAGES).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    save_prediction_figures(best_model, [(p, None) for p in ext[:N_PRED_FIGS]], "external")

# Cell 7: list outputs
for p in sorted(OUT.rglob("*")):
    if p.is_file() and p.suffix in {".csv", ".json"}:
        print(p.relative_to(WORK))
print("PNG figures:", len(list(PNG_DIR.glob('*.png'))), "| EPS figures:", len(list(EPS_DIR.glob('*.eps'))))
print("Weights:", BEST, "and", LAST)